# **Mortalité des enfants**
# Données issues des Enquêtes Démographiques et de Santé (EDS - DHS)

Ce rapport génère des visualisations pour l'indicateur relatif à la **mortalité générale des enfants âgés de 0 à 4 ans**, à partir des données de l'Enquête Démographique et de Santé (EDS - DHS).

Le taux de mortalité des moins de 5 ans (TM5) correspond à la probabilité (exprimée pour 1 000 naissances vivantes) qu'un enfant né au cours d'une période donnée décède avant d'avoir atteint l'âge de cinq ans. L'indicateur est calculé sur base des naissances vivantes des personnes interrogées, durant les dix années précédant l'enquête.

---

* *Numérateur* : Décès chez les enfants âgés de 0 à 4 ans, y compris les décès déclarés chez les enfants âgés de 0 à 59 mois et de 0 à 99 jours.
* *Dénominateur* : Nombre d'enfants en vie au début de la tranche d'âge, au cours de la période de référence.

---

Pour plus d'informations (en anglais):
- Ressources relatives à la mortalité des enfants
    - [Définition et calculs](https://dhsprogram.com/data/Guide-to-DHS-Statistics/index.htm#t=Early_Childhood_Mortality.htm)
- [Les questionnaires utilisés dans les EDS/DHS](https://dhsprogram.com/publications/publication-dhsg4-dhs-questionnaires-and-manuals.cfm)

---

*Note* : Contrairement à la majorité des analyses dans le cadre du processus SNT, cette analyse est menée au niveau administratif **ADM1**, en raison de la disponibilité des données.

## 1. Configuration

In [ ]:
rm(list = ls())

options(scipen=999)

# Global paths
Sys.setenv(PROJ_LIB = "/opt/conda/share/proj")
Sys.setenv(GDAL_DATA = "/opt/conda/share/gdal")

# Paths
ROOT_PATH <- '~/workspace'
PIPELINE_PATH <- file.path(ROOT_PATH, 'pipelines', 'snt_dhs_indicators')
CONFIG_PATH <- file.path(ROOT_PATH, 'configuration')
CODE_PATH <- file.path(ROOT_PATH, 'code')
DATA_PATH <- file.path(ROOT_PATH, 'data')
DHS_DATA_PATH <- file.path(DATA_PATH, 'dhs', 'raw')
OUTPUT_DATA_PATH <- file.path(DATA_PATH, 'dhs', 'indicators', 'mortality')
OUTPUT_PLOTS_PATH <- file.path(ROOT_PATH, 'pipelines', 'snt_dhs_indicators', 'reporting', 'outputs')


In [ ]:
# Load notebook-specific utilities
source(file.path(CODE_PATH, "snt_utils.r"))
source(file.path(CODE_PATH, "snt_report.r"))
source(file.path(CODE_PATH, "snt_palettes.r"))
source(file.path(PIPELINE_PATH, "utils", "snt_dhs_mortality_report.r"))

# List required pcks
required_packages <- c("sf", "glue", "data.table", "ggplot2", "stringi", "jsonlite", "httr", "reticulate", "arrow", "IRdisplay")

# Execute function
install_and_load(required_packages)

Sys.setenv(RETICULATE_PYTHON = "/opt/conda/bin/python")
reticulate::py_config()$python
openhexa <- import("openhexa.sdk")

# Load SNT config
CONFIG_FILE_NAME <- "SNT_config.json"
config_json <- tryCatch({ fromJSON(file.path(CONFIG_PATH, CONFIG_FILE_NAME)) },
                        error = function(e) {
                          msg <- paste0("Error while loading configuration", conditionMessage(e))
                          cat(msg)
                          stop(msg)
                        })

msg <- paste0("SNT configuration : ", file.path(CONFIG_PATH, CONFIG_FILE_NAME))
log_msg(msg)

# Set config variables
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE

data_source <- 'DHS'

## 2. Chargement et pré-processing des données à visualiser

**Les données utilisées**

Toutes les données utilisées dans ce rapport sont agrégées au niveau administratif ADM1:

* Données spatiales : fond de carte (DHIS2)
* Données calculées par le pipeline :
    - valeurs estimées du taux de mortalité des enfants de moins de 5 ans, ainsi que sur la précision statistique de cette estimation (intervalles de confiance à 95%)

In [ ]:
# Spatial data

admin_level <- 'ADM1'
admin_id_col <- glue(admin_level, 'ID', .sep='_')
admin_name_col <- glue(admin_level, 'NAME', .sep='_')
admin_cols <- c(admin_id_col, admin_name_col)

# Load spatial file from dataset

dhis2_dataset <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED

spatial_data_filename <- paste(COUNTRY_CODE, "shapes.geojson", sep = "_")
# spatial_data <- read_sf(file.path(DATA_PATH, 'dhis2', 'formatted', spatial_data_filename))
spatial_data <- get_latest_dataset_file_in_memory(dhis2_dataset, spatial_data_filename)
log_msg(glue("File {spatial_data_filename} successfully loaded from dataset version: {dhis2_dataset}"))

spatial_data <- st_as_sf(spatial_data)

# aggregate geometries by the admin columns
spatial_data <- aggregate_geometry(
  sf_data=spatial_data,
  admin_id_colname=admin_id_col,
  admin_name_colname=admin_name_col
)

# keep class
spatial_data <- st_as_sf(spatial_data)

if(COUNTRY_CODE == "COD"){
  spatial_data[[admin_name_col]] <- clean_admin_names(spatial_data[[admin_name_col]])
}

## 3. Création de graphiques

Le rapport crée deux visualisations:
   - Une carte choroplèthe indiquant l'estimation moyenne de la valeur de l'indicateur, sur base des données d'enquête
   - Un graphique de son intervalle de confiance, avec des barres d’erreur pour chaque ADM1

In [ ]:
indicator_u5mr <- 'U5MR_PERMIL'
lower_bound_col <- glue("{toupper(indicator_u5mr)}_CI_LOWER_BOUND")
upper_bound_col <- glue("{toupper(indicator_u5mr)}_CI_UPPER_BOUND")
sample_avg_col <- glue("{toupper(indicator_u5mr)}_SAMPLE_AVERAGE")

filename_without_extension <- glue("{COUNTRY_CODE}_{data_source}_{admin_level}_{toupper(indicator_u5mr)}")
u5mort_table <- fread(file.path(OUTPUT_DATA_PATH, paste0(filename_without_extension, '.csv')))

In [ ]:
plot_data =  merge(spatial_data, u5mort_table, by = admin_cols, all = TRUE)

In [ ]:
mort_plot <- make_permil_choropleth_map(
  map_data = plot_data,
  target_colname = sample_avg_col,
  plot_title = "Mortalité des enfants (TM5, \u2030)",
  plot_subtitle = COUNTRY_CODE,
  plot_caption = glue("Données: {data_source}"),
  scale_range = c(0, 200) # il s'agit d'un taux pour mille
)


In [ ]:
mort_plot_filename <- glue("{COUNTRY_CODE}_{data_source}_{admin_level}_{toupper(indicator_u5mr)}_plot.png")
mort_plot_path <- file.path(OUTPUT_PLOTS_PATH, mort_plot_filename)
suppressMessages(ggsave(mort_plot, file = mort_plot_path, width = 6, height = 5, dpi = 300))

In [ ]:
display_png(file = mort_plot_path)

In [ ]:
mort_ci_plot_title <- glue("Mortalité des enfants (TM5 - Intervalles de Confiance 95%)")
mort_ci_plot_xlab <- admin_level
mort_ci_plot_ylab <- glue("TM5 (\u2030)")

mort_ci_plot <- make_ci_plot(
  df_to_plot=plot_data,
  admin_colname=admin_name_col,
  point_estimation_colname=sample_avg_col,
  ci_lower_colname=lower_bound_col,
  ci_upper_colname=upper_bound_col,
  plot_title=mort_ci_plot_title,
  plot_subtitle=COUNTRY_CODE,
  plot_caption=glue("Données: {data_source}"),
  x_title=mort_ci_plot_xlab,
  y_title=mort_ci_plot_ylab
)

In [ ]:
mort_ci_plot_filename <- glue("{COUNTRY_CODE}_{data_source}_{admin_level}_{toupper(indicator_u5mr)}_CI_plot.png")
mort_ci_plot_path <- file.path(OUTPUT_PLOTS_PATH, mort_ci_plot_filename)
suppressMessages(ggsave(filename=mort_ci_plot_path, plot=mort_ci_plot, width = 6, height = 5, dpi = 300))

In [ ]:
display_png(file = mort_ci_plot_path)